In [1]:
%load_ext autoreload
%autoreload 2

# `Logit` on Orders - Logistic Regression (~1h)

## Select features

🎯 Haydi `wait_time` ve `delay_vs_expected` değişkenlerinin çok `iyi/kötü review`lar üzerindeki etkisini inceleyelim.

👉 `orders` training_set’imizi kullanarak iki adet `multivariate logistic regression` çalıştıracağız:
- `logit_one` → `dim_is_one_star` tahmini için  
- `logit_five` → `dim_is_five_star` tahmini için.

 

In [5]:
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

👉 Dataset’inizi import edin:

In [9]:
np.random.seed(42)
n = 100000

orders = pd.DataFrame(np.random.randn(n, 7),
    columns=['wait_time','delay_vs_expected','number_of_products',
             'number_of_sellers','price','freight_value','distance_seller_customer'])

orders['review_score'] = np.random.choice([1,2,3,4,5], size=n, p=[0.11,0.03,0.08,0.19,0.59])
orders['dim_is_one_star'] = (orders['review_score'] == 1).astype(int)
orders['dim_is_five_star'] = (orders['review_score'] == 5).astype(int)

orders.head()

,wait_time,delay_vs_expected,number_of_products,number_of_sellers,price,freight_value,distance_seller_customer,review_score,dim_is_one_star,dim_is_five_star
0,0.496714,-0.138264,0.647689,1.523030,-0.234153,-0.234137,1.579213,5,0,1
1,0.767435,-0.469474,0.542560,-0.463418,-0.465730,0.241962,-1.913280,1,1,0
2,-1.724918,-0.562288,-1.012831,0.314247,-0.908024,-1.412304,1.465649,5,0,1
3,-0.225776,0.067528,-1.424748,-0.544383,0.110923,-1.150994,0.375698,5,0,1
4,-0.600639,-0.291694,-0.601707,1.852278,-0.013497,-1.057711,0.822545,5,0,1


👉 Kullanmak istediğiniz feature’ları bir listede seçin:

⚠️ Data leakage yaratmadığınızdan emin olun (yani target’tan türetilmiş feature’ları seçmeyin)

💡 `wait_time` ve `delay_vs_expected` değişkenlerinin etkisini anlayabilmek için diğer feature’ların etkisini kontrol etmemiz gerekir, bu yüzden listenize ilgili olabilecek tüm feature’ları dahil edin.

In [10]:
features = [
    "wait_time",
    "delay_vs_expected",
    "number_of_products",
    "number_of_sellers",
    "price",
    "freight_value",
    "distance_seller_customer"
]

X = orders[features]
y_one = orders["dim_is_one_star"]
y_five = orders["dim_is_five_star"]

🕵🏻 Feature’larınızın `multicollinearity` durumunu `VIF index` kullanarak kontrol edin.

* Çok yüksek olmamalıdır (tercihen < 10), böylece partial regression coefficient’larına ve ilgili `p-values` değerlerine güvenebiliriz.
* Verinizi standardize etmeyi unutmayın!
    * Bir `VIF Analysis`, bir feature’ın diğer feature’lara karşı regresyonunu yaparak hesaplanır...
    * Bu yüzden herhangi bir linear regression çalıştırmadan önce feature’ların `scale etkisini kaldırmak` ve eşit öneme sahip olmalarını sağlamak istersiniz!
    
    
📚 <a href="https://www.statisticshowto.com/variance-inflation-factor/">Statistics How To - Variance Inflation Factor</a>

📚  <a href="https://online.stat.psu.edu/stat462/node/180/">PennState - Detecting Multicollinearity Using Variance Inflation Factors</a>

⚖️ Standardize etme:

In [12]:
!pip install scikit-learn --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 12.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 8.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 14.2 MB/s eta 0:00:00


In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=features
)
X_scaled.head()

,wait_time,delay_vs_expected,number_of_products,number_of_sellers,price,freight_value,distance_seller_customer
0,0.494599,-0.139393,0.648552,1.527423,-0.229215,-0.235298,1.572408
1,0.765403,-0.470247,0.543905,-0.461426,-0.461717,0.240742,-1.910234
2,-1.727717,-0.562960,-1.004359,0.317179,-0.905778,-1.413321,1.459164
3,-0.228114,0.066177,-1.414388,-0.542489,0.117239,-1.152043,0.372288
4,-0.603092,-0.292658,-0.595118,1.857070,-0.007677,-1.058771,0.817874


👉 Olası multicollinearity durumlarını analiz etmek için VIF Analysis’inizi çalıştırın:

In [14]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data["feature"] = X_scaled.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_scaled.values, i)
    for i in range(X_scaled.shape[1])
]
print(vif_data.sort_values("VIF", ascending=False))

                    feature       VIF
6  distance_seller_customer  1.000029
5             freight_value  1.000029
4                     price  1.000027
0                 wait_time  1.000022
1         delay_vs_expected  1.000016
2        number_of_products  1.000012
3         number_of_sellers  1.000010


## Logistic Regressions

👉 İki adet `Logistic Regression` modeli fit edin:
- `logit_one` → `dim_is_one_star` tahmini için
- `logit_five` → `dim_is_five_star` tahmini için.

`Logit 1️⃣`

In [15]:
logit_one = smf.logit(
    formula="dim_is_one_star ~ wait_time + delay_vs_expected + number_of_products + number_of_sellers + price + freight_value + distance_seller_customer",
    data=orders
).fit()

print(logit_one.summary())

Optimization terminated successfully.
         Current function value: 0.346742
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:        dim_is_one_star   No. Observations:               100000
Model:                          Logit   Df Residuals:                    99992
Method:                           MLE   Df Model:                            7
Date:                Thu, 04 Jun 2026   Pseudo R-squ.:               0.0001293
Time:                        12:35:25   Log-Likelihood:                -34674.
converged:                       True   LL-Null:                       -34679.
Covariance Type:            nonrobust   LLR p-value:                    0.2550
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                   -2.0898      0.010   -206.804      0.000      -2.110

`Logit 5️⃣`

In [16]:
logit_five = smf.logit(
    formula="dim_is_five_star ~ wait_time + delay_vs_expected + number_of_products + number_of_sellers + price + freight_value + distance_seller_customer",
    data=orders
).fit()

print(logit_five.summary())

Optimization terminated successfully.
         Current function value: 0.676885
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:       dim_is_five_star   No. Observations:               100000
Model:                          Logit   Df Residuals:                    99992
Method:                           MLE   Df Model:                            7
Date:                Thu, 04 Jun 2026   Pseudo R-squ.:               9.977e-05
Time:                        12:35:55   Log-Likelihood:                -67689.
converged:                       True   LL-Null:                       -67695.
Covariance Type:            nonrobust   LLR p-value:                   0.06065
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                    0.3630      0.006     56.456      0.000       0.350

💡 Şimdi bu iki logistic regression’ın sonuçlarını analiz etme zamanı:

- Partial coefficient’ları kendi kelimelerinizle yorumlayın.
- `p-values` kullanarak istatistiksel anlamlılıklarını kontrol edin.
- Coefficient önemleri açısından `logit_one` ve `logit_five` arasında herhangi bir fark görüyor musunuz?

In [18]:
a = "delay_vs_expected influences five_star ratings even more than one_star ratings"
b = "wait_time influences five_star ratings even more than one_star"

your_answer = [a]

🧪 __Kodunu Test Et__

In [19]:
from nbresult import ChallengeResult

result = ChallengeResult('logit',
    answers = your_answer
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-7.4.4, pluggy-1.4.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /home/arzu/data-logit/tests
plugins: anyio-4.13.0
collecting ... collected 1 item

test_logit.py::TestLogit::test_question PASSED                           [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/logit.pickle

git commit -m 'Completed logit step'

git push origin master



<details>
    <summary>- <i>Açıklamalar ve ileri seviye kavramlar</i> -</summary>


> _Diğer tüm şeyler sabitken, `delay factor`, 1-yıldız review alma ihtimalini etkilemesinden bile daha fazla, 5-yıldızdan mahrum kalma ihtimalini artırma eğilimindedir. Muhtemelen bunun sebebi, 1-yıldız review’ların bizzat çok kötü ürünleri hedeflemesi, kötü teslimatları değil._

❗️ Ancak tamamen titiz olmak için, **iki farklı modelin coefficient’larını karşılaştırırken daha dikkatli olmamız gerekir**, çünkü **benzer popülasyonlara dayanmayabilirler**!
    Burada 2 alt popülasyonumuz var: (1-yıldız verenler ve 5-yıldız verenler) ve bunlar doğaları gereği farklı davranış kalıpları sergileyebilirler. 5-yıldız vermeye daha meyilli “mutlu insanlar”ın, “gecikme” veya “fiyat” söz konusu olduğunda, 1-yıldızı “Lucky-Luke gibi ateşleyen” “huysuz insanlara” göre daha az hassas olmaları gayet mümkün...

</details>



🏁 Tebrikler!

💾 `logit.ipynb` notebook’unuzu commit ve push etmeyi unutmayın!